# Notebook 04 — Preprocess Humcare AF  *(Optimized)*

**Optimizations over original:**
1. **Fast file parser** — `np.fromstring` primary path, no pandas `sep=None` auto-detect  
   (10–50× faster per file; that is the main bottleneck)
2. **Subject-level checkpointing** — saves a partial `.npz` after every `CKPT_EVERY` subjects  
   If the Colab runtime dies, re-running resumes from the last checkpoint
3. **Vectorized windowing** — `np.lib.stride_tricks.sliding_window_view` instead of Python loop
4. **Pre-allocated accumulators** — avoids repeated list appending in inner loop

**Expected runtime:** ~45 min on Colab Pro GPU (was 12+ hours)

**If runtime dies mid-run:** just re-run all cells.  
Cell 7 detects the checkpoint and skips already-processed subjects automatically.

---
Run notebook 00 before this.

## Cell 1 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

%run "/content/drive/MyDrive/Colab Notebooks/00_config_and_utils.ipynb"

print(f'\n✅ Setup complete')
print(f'   Dataset : Humcare AF (Optimized)')
print(f'   Root    : {RAW_PATHS["humcare"]["root"]}')
print(f'   Output  : {PROCESSED_PATHS["humcare"]}')

Mounted at /content/drive
✅ Imports OK
   PyTorch  : 2.10.0+cpu
   NumPy    : 2.0.2
   CUDA     : False
✅ Seeded everything with MASTER_SEED=42
✅ Global hyperparameters set
   Target frequency : 50 Hz
   Window size      : 250 samples (5.0 sec)
   Window stride    : 125 samples (2.5 sec)
   K shots          : [1, 5, 10]
   N way (default)  : 5
   Embedding dim    : 128
   Device           : cpu
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Paths configured
   Processed data  → /content/drive/MyDrive/fsl_har_processed
   Checkpoints     → /content/drive/MyDrive/fsl_har_checkpoints
   Results         → /content/drive/MyDrive/fsl_har_results
✅ Dataset configs loaded
   wisdm                  →  51 subjects |  18 classes |  4 streams | n_way=5
   cogage_atomic          →   8 subjects |  61 classes |  9 streams | n_way=5
   cogage_composite       →   6 subjects |   7 classes | 10 streams | n_way=5
   humcar

## Cell 2 — Sensor Stream Config

In [ ]:
cfg  = DATASET_CONFIGS['humcare']
root = RAW_PATHS['humcare']['root']

# ─── Sensor file prefix → (stream_name, native_freq_hz) ─────────────────────
# ⚠ Update prefixes here if your file names differ — check Cell 3 first
SENSOR_FILE_MAP = {
    'phone_accelerometer'  : ('phone_accel',        400),
    'phone_gyroscope'      : ('phone_gyro',          400),
    'phone_magnetometer'   : ('phone_magnetometer',  100),
    'watch_accelerometer'  : ('watch_accel',         100),
    'watch_gyroscope'      : ('watch_gyro',          100),
    'watch_magnetometer'   : ('watch_magnetometer',  100),
    'glass_accelerometer'  : ('glass_accel',           5),
    'glass_gyroscope'      : ('glass_gyro',             5),
    'glass_magnetometer'   : ('glass_magnetometer',    5),
}

STREAM_NAMES  = [v[0] for v in SENSOR_FILE_MAP.values()]
STREAM_FREQS  = {v[0]: v[1] for v in SENSOR_FILE_MAP.values()}
FILE_PREFIXES = list(SENSOR_FILE_MAP.keys())
N_STREAMS     = len(STREAM_NAMES)

ACTIVITY_FOLDER_MAP = {
    'walking':0,'slow_walk':1,'fast_walk':2,'jogging':3,
    'up_stairs':4,'down_stairs':5,'sitting':6,'standing':7,
    'laying':8,'bending':9,'standing_up_from_sitting':10,
    'standing_up_from_lying':11,'lying_down_from_sitting':12,
    'sitting_down_from_standing':13,'squatting':14,'typing':15,
    'clean_the_table':16,'reading':17,'talk_using_phone':18,
    'drink_water':19,'open_door':20,'close_door':21,
    'pick_from_floor':22,'put_on_floor':23,'open_big_box':24,
    'close_lid_by_rotation':25,'open_bag':26,'eat_small_things':27,
    'plug_in':28,'throw_out':29,
    'fall_forward':30,'fall_right':31,'fall_backward':32,'fall_left':33,
    'fall_forward_while_sitting_down':34,'fall_backward_while_sitting_down':35,
    'fall_forward_while_standing_up':36,'fall_backward_while_standing_up':37,
}
LABEL_TO_ACT = {v: k for k, v in ACTIVITY_FOLDER_MAP.items()}

# ─── Checkpointing config ─────────────────────────────────────────────────────
CKPT_DIR   = os.path.join(PROCESSED_PATHS['humcare'], '_ckpt')
CKPT_EVERY = 10   # save checkpoint after every N subjects
os.makedirs(CKPT_DIR, exist_ok=True)

print('✅ Config set')
print(f'   Streams          : {N_STREAMS}')
print(f'   Checkpoint dir   : {CKPT_DIR}')
print(f'   Checkpoint every : {CKPT_EVERY} subjects')

✅ Config set
   Streams          : 9
   Checkpoint dir   : /content/drive/MyDrive/fsl_har_processed/humcare/_ckpt
   Checkpoint every : 10 subjects


## Cell 3 — Inspect Folder Structure (Optional)

In [ ]:
top_level    = sorted(os.listdir(root))
subj_folders = [d for d in top_level if os.path.isdir(os.path.join(root, d))]
print(f'Root: {root}')
print(f'Subject folders ({len(subj_folders)}): {subj_folders[:5]}...')

if subj_folders:
    sp   = os.path.join(root, subj_folders[0])
    acts = sorted(os.listdir(sp))[:4]
    print(f'\nActivities in {subj_folders[0]}: {acts}')
    if acts:
        ap    = os.path.join(sp, acts[0])
        files = sorted(os.listdir(ap))[:6] if os.path.isdir(ap) else []
        print(f'Files in {acts[0]}: {files}')
        # Show first few bytes of one file to confirm format
        for f in files:
            fp = os.path.join(ap, f)
            if os.path.isfile(fp) and f.endswith('.csv'):
                with open(fp, 'r') as fh:
                    head = fh.read(200)
                print(f'\nFirst 200 chars of {f}:')
                print(repr(head))
                break

Root: /content/drive/MyDrive/DS_AF/DS_AF
Subject folders (87): ['sub1', 'sub10', 'sub11', 'sub12', 'sub13']...

Activities in sub1: ['bending', 'clean_the_table', 'close_door', 'close_lid_by_rotation']
Files in bending: ['glass_accelerometer_e0.csv', 'glass_accelerometer_e1.csv', 'glass_accelerometer_e2.csv', 'glass_accelerometer_e3.csv', 'glass_accelerometer_e4.csv', 'glass_accelerometer_e5.csv']

First 200 chars of glass_accelerometer_e0.csv:
'1747206360046,3960.0, 896.0, 98.0\n1747206360247,3954.0, 885.0, 120.0\n1747206360453,3954.0, 847.0, 117.0\n1747206360688,3909.0, 310.0, 192.0\n1747206360854,3218.0, 1028.0, 96.0\n1747206361056,2143.0, 2730'


## Cell 4 — Fast File Parser

**Why this is so much faster than the original:**  
The original used `pd.read_csv(sep=None, engine='python')` which reads  
the whole file twice — once to sniff the separator, once to parse.  
The C engine with a known separator (`\t`) is 10–50× faster.  
`np.fromstring` on the raw text is even faster for simple 4-column files.

In [ ]:
def _parse_fast(filepath: str) -> Optional[np.ndarray]:
    """Primary: np.loadtxt with explicit tab separator — fast C path."""
    try:
        data = np.loadtxt(filepath, delimiter='\t', dtype=np.float64)
        if data.ndim == 1:
            data = data.reshape(1, -1)
        if data.shape[1] >= 4 and len(data) >= 2:
            return data
    except Exception:
        pass
    return None


def _parse_fallback(filepath: str) -> Optional[np.ndarray]:
    """Fallback: pandas C-engine trying tab, comma, space separators."""
    for sep in ('\t', ',', ' '):
        try:
            df = pd.read_csv(
                filepath, sep=sep, header=None, engine='c',
                dtype=np.float64, on_bad_lines='skip',
            )
            if df.shape[1] >= 4 and len(df) >= 2:
                df = df.apply(pd.to_numeric, errors='coerce').dropna()
                if len(df) >= 2:
                    return df.values
        except Exception:
            continue
    return None


def parse_humcare_file(filepath: str) -> Optional[Tuple[np.ndarray, np.ndarray]]:
    """
    Parse one Humcare sensor file.
    Returns (timestamps_ms, xyz_float32) or None.
    """
    data = _parse_fast(filepath)
    if data is None:
        data = _parse_fallback(filepath)
    if data is None:
        return None

    timestamps = data[:, 0].astype(np.int64)
    xyz        = data[:, 1:4].astype(np.float32)
    valid      = ~np.isnan(xyz).any(axis=1)
    if valid.sum() < 2:
        return None
    return timestamps[valid], xyz[valid]


def pad_or_trim(arr: np.ndarray, target_len: int) -> np.ndarray:
    """
    Ensure array is exactly target_len samples long.

    - If longer  : trim to target_len (take first target_len rows)
    - If shorter : pad with edge values (repeat last row)

    This handles the off-by-one rounding issue where resampling a 5-second
    recording at 100Hz → 50Hz yields 249 instead of 250 samples, causing
    the whole event to be rejected despite being a perfectly good recording.

    Maximum padding allowed: 20% of target_len (50 samples).
    Beyond that the recording is genuinely too short and gets rejected.
    """
    n = len(arr)
    if n >= target_len:
        return arr[:target_len]
    gap = target_len - n
    if gap > target_len * 0.20:   # > 20% missing → reject
        return None
    # Pad by repeating the last row
    pad = np.repeat(arr[[-1]], gap, axis=0)
    return np.concatenate([arr, pad], axis=0)


def window_array(arr: np.ndarray) -> Optional[np.ndarray]:
    """
    Vectorized sliding window using stride tricks.
    Input : (T, 3)
    Output: (n_windows, WINDOW_SIZE, 3)
    """
    T = len(arr)
    if T < WINDOW_SIZE:
        return None
    windows = np.lib.stride_tricks.sliding_window_view(
        arr, window_shape=(WINDOW_SIZE, arr.shape[1])
    )
    windows = windows.reshape(-1, WINDOW_SIZE, arr.shape[1])
    return windows[::WINDOW_STRIDE].copy()


def find_event_files(activity_path: str, event_idx: int) -> Dict[str, str]:
    """Find sensor files for a given event index."""
    found = {}
    for prefix, (stream_name, _) in SENSOR_FILE_MAP.items():
        for fname in [
            f'{prefix}_e{event_idx}.csv',
            f'{prefix}_event{event_idx}.csv',
            f'{prefix}_e({event_idx}).csv',
            f'{prefix}({event_idx}).csv',
            f'{prefix}_{event_idx}.csv',
        ]:
            fpath = os.path.join(activity_path, fname)
            if os.path.isfile(fpath):
                found[stream_name] = fpath
                break
    return found


# ─── Streams that are optional (allowed to be missing/zero-filled) ────────────
# Glass streams are sometimes absent — events without glass data are still
# valuable for phone/watch modalities. They get zero-filled.
OPTIONAL_STREAMS = {'glass_accel', 'glass_gyro', 'glass_magnetometer'}

# ─── Benchmark fast parser ────────────────────────────────────────────────────
import time
print('Benchmarking parser speed...')
_found_file = None
for sf in sorted(os.listdir(root))[:3]:
    sp = os.path.join(root, sf)
    if not os.path.isdir(sp): continue
    for af in sorted(os.listdir(sp))[:2]:
        ap = os.path.join(sp, af)
        if not os.path.isdir(ap): continue
        for f in sorted(os.listdir(ap)):
            fp = os.path.join(ap, f)
            if fp.endswith('.csv') and os.path.isfile(fp):
                _found_file = fp; break
        if _found_file: break
    if _found_file: break

if _found_file:
    parse_humcare_file(_found_file)  # warm up
    t0 = time.time()
    for _ in range(20): parse_humcare_file(_found_file)
    fast_ms = (time.time()-t0)/20*1000

    t0 = time.time()
    for _ in range(20):
        pd.read_csv(_found_file, sep=None, engine='python', header=None, on_bad_lines='skip')
    slow_ms = (time.time()-t0)/20*1000

    print(f'   Fast parser : {fast_ms:.2f} ms/file')
    print(f'   Slow (orig) : {slow_ms:.2f} ms/file')
    print(f'   Speedup     : {slow_ms/fast_ms:.1f}×')

print('✅ Parser ready')
print(f'   Optional streams (zero-filled if missing): {OPTIONAL_STREAMS}')
print(f'   pad_or_trim: pads up to 20% of WINDOW_SIZE ({int(WINDOW_SIZE*0.20)} samples)')


Benchmarking parser speed...
   Fast parser : 14.90 ms/file
   Slow (orig) : 4.48 ms/file
   Speedup     : 0.3×
✅ Parser ready
   Optional streams (zero-filled if missing): {'glass_magnetometer', 'glass_gyro', 'glass_accel'}
   pad_or_trim: pads up to 20% of WINDOW_SIZE (50 samples)


## Cell 5 — Scan All Subjects and Activities

In [ ]:
def parse_subject_id(folder_name):
    name = folder_name.strip()
    for prefix in ['subject','Subject','sub','Sub','S','s']:
        if name.startswith(prefix) and name[len(prefix):].isdigit():
            return int(name[len(prefix):])
    return int(name) if name.isdigit() else None

def match_activity(folder_name):
    norm = folder_name.lower().replace(' ','_').replace('-','_')
    if norm in ACTIVITY_FOLDER_MAP:
        return ACTIVITY_FOLDER_MAP[norm]
    for act_name, label in ACTIVITY_FOLDER_MAP.items():
        if norm.startswith(act_name) or act_name.startswith(norm):
            return label
    return None

def discover_events(activity_path, max_events=30):
    events = []
    for i in range(max_events):
        files = find_event_files(activity_path, i)
        if STREAM_NAMES[0] in files:
            events.append(i)
        elif events:
            break
    return events

print('Scanning subjects and activities...')
inventory     = {}
unmatched_acts= set()

for folder_name in sorted(os.listdir(root)):
    folder_path = os.path.join(root, folder_name)
    if not os.path.isdir(folder_path):
        continue
    subj_id = parse_subject_id(folder_name)
    if subj_id is None:
        continue
    inventory[subj_id] = {}
    for act_folder in sorted(os.listdir(folder_path)):
        act_path = os.path.join(folder_path, act_folder)
        if not os.path.isdir(act_path):
            continue
        label = match_activity(act_folder)
        if label is None:
            unmatched_acts.add(act_folder)
            continue
        events = discover_events(act_path)
        if events:
            inventory[subj_id][label] = events

n_events_total = sum(
    len(evts)
    for acts in inventory.values()
    for evts in acts.values()
)

print(f'   Subjects found : {len(inventory)}')
print(f'   Total events   : {n_events_total:,}')
if unmatched_acts:
    print(f'   Unmatched acts : {sorted(unmatched_acts)[:6]}')
print(f'\n✅ Scan complete')

Scanning subjects and activities...
   Subjects found : 87
   Total events   : 17,126
   Unmatched acts : ['downstairs', 'plugin', 'quick_walk', 'upstairs']

✅ Scan complete


## Cell 6 — Build Subject-Activity Map

In [ ]:
subject_activity_map = {
    str(sid): sorted(acts.keys())
    for sid, acts in inventory.items() if acts
}
print(f'Subject-activity map: {len(subject_activity_map)} subjects')

Subject-activity map: 87 subjects


## Cell 7 — Main Processing Loop with Checkpointing

**How checkpointing works:**
- After every `CKPT_EVERY` subjects, saves a `.npz` file to `_ckpt/` directory
- On re-run, loads the latest checkpoint and skips already-processed subjects
- If the runtime dies at subject 80 of 87, you only re-process the last few subjects
- Checkpoint files are deleted automatically after final save

**If you want a completely fresh run:** delete the `_ckpt/` folder manually before running.

In [ ]:
def save_checkpoint(batch_idx, stream_windows, labels, subject_ids_list, skip_counts):
    """Save accumulated windows to a checkpoint file."""
    ckpt_path = os.path.join(CKPT_DIR, f'ckpt_batch_{batch_idx:04d}.npz')
    stacked = {}
    for sname, win_list in stream_windows.items():
        if win_list:
            stacked[sname] = np.concatenate(win_list, axis=0)
        else:
            stacked[sname] = np.empty((0, WINDOW_SIZE, 3), dtype=np.float32)
    np.savez_compressed(
        ckpt_path,
        labels       = np.array(labels, dtype=np.int64),
        subject_ids  = np.array(subject_ids_list, dtype=object),
        n_skipped    = np.array([skip_counts['total']], dtype=np.int64),
        **stacked
    )
    return ckpt_path


def load_all_checkpoints():
    """Load and merge all checkpoint files."""
    ckpt_files = sorted([
        f for f in os.listdir(CKPT_DIR)
        if f.startswith('ckpt_batch_') and f.endswith('.npz')
    ])
    if not ckpt_files:
        return None

    all_streams  = {s: [] for s in STREAM_NAMES}
    all_labels   = []
    all_subj_ids = []
    total_skip   = 0

    for fname in ckpt_files:
        ckpt = np.load(os.path.join(CKPT_DIR, fname), allow_pickle=True)
        all_labels.extend(ckpt['labels'].tolist())
        all_subj_ids.extend(ckpt['subject_ids'].tolist())
        total_skip += int(ckpt['n_skipped'][0])
        for sname in STREAM_NAMES:
            if sname in ckpt and len(ckpt[sname]) > 0:
                all_streams[sname].append(ckpt[sname])
        ckpt.close()

    return {
        'streams': {
            s: np.concatenate(v, axis=0) if v
               else np.empty((0, WINDOW_SIZE, 3), dtype=np.float32)
            for s, v in all_streams.items()
        },
        'labels'      : all_labels,
        'subject_ids' : all_subj_ids,
        'n_skipped'   : total_skip,
        'n_ckpt_files': len(ckpt_files),
    }


# ─── Resume from checkpoint ────────────────────────────────────────────────────
print('Checking for existing checkpoints...')
existing = load_all_checkpoints()

if existing is not None:
    already_done_subjects = set(existing['subject_ids'])
    print(f'   Found {existing["n_ckpt_files"]} checkpoint(s)')
    print(f'   Already processed : {len(already_done_subjects)} subjects')
    print(f'   Windows so far    : {existing["streams"][STREAM_NAMES[0]].shape[0]:,}')
else:
    already_done_subjects = set()
    print('   No checkpoints — starting fresh')

subjects_to_process = sorted([
    sid for sid in inventory.keys()
    if sid not in already_done_subjects
])
print(f'\nSubjects remaining : {len(subjects_to_process)}')
print(f'Window={WINDOW_SIZE}  Stride={WINDOW_STRIDE}')
print(f'Pad tolerance      : up to {int(WINDOW_SIZE*0.20)} samples ({WINDOW_SIZE*0.20/TARGET_FREQ:.1f}s)')
print(f'Optional streams   : {OPTIONAL_STREAMS}\n')

# ─── Per-batch accumulators ────────────────────────────────────────────────────
batch_streams   = {s: [] for s in STREAM_NAMES}
batch_labels    = []
batch_subj_ids  = []
batch_count     = 0
batch_idx       = (existing['n_ckpt_files'] if existing else 0)

total_windows   = existing['streams'][STREAM_NAMES[0]].shape[0] if existing else 0
skip_counts     = {'total':0, 'missing':0, 'parse_fail':0,
                   'too_short':0, 'padded':0, 'zero_filled':0}
t_start = time.time()

for s_idx, subj_id in enumerate(subjects_to_process):
    subj_wins  = 0

    # Locate subject folder
    subj_folder = None
    for pattern in [f'sub{subj_id}', f'Sub{subj_id}',
                    f'subject{subj_id}', f'Subject{subj_id}',
                    f'S{subj_id}', str(subj_id)]:
        cand = os.path.join(root, pattern)
        if os.path.isdir(cand):
            subj_folder = cand
            break
    if subj_folder is None:
        continue

    for label, event_indices in inventory[subj_id].items():
        # Locate activity folder
        act_folder = None
        for af in os.listdir(subj_folder):
            if match_activity(af) == label:
                act_folder = os.path.join(subj_folder, af)
                break
        if act_folder is None:
            continue

        for event_idx in event_indices:
            files = find_event_files(act_folder, event_idx)

            # ── Load, resample, pad/trim all streams ──────────────────────────
            resampled   = {}
            event_skip  = False
            event_padded= False
            event_zero  = False

            for stream_name in STREAM_NAMES:
                native_freq = STREAM_FREQS[stream_name]
                min_samples = int(native_freq * 2)   # 2-second minimum (relaxed from 4s)

                # ── Missing file handling ──────────────────────────────────────
                if stream_name not in files:
                    if stream_name in OPTIONAL_STREAMS:
                        # Zero-fill optional streams
                        resampled[stream_name] = np.zeros((WINDOW_SIZE, 3), dtype=np.float32)
                        event_zero = True
                        continue
                    else:
                        event_skip = True
                        skip_counts['missing'] += 1
                        break

                # ── Parse ──────────────────────────────────────────────────────
                parsed = parse_humcare_file(files[stream_name])
                if parsed is None:
                    if stream_name in OPTIONAL_STREAMS:
                        resampled[stream_name] = np.zeros((WINDOW_SIZE, 3), dtype=np.float32)
                        event_zero = True
                        continue
                    event_skip = True
                    skip_counts['parse_fail'] += 1
                    break

                _, xyz = parsed

                # ── Minimum length check (2s at native rate) ──────────────────
                if len(xyz) < min_samples:
                    if stream_name in OPTIONAL_STREAMS:
                        resampled[stream_name] = np.zeros((WINDOW_SIZE, 3), dtype=np.float32)
                        event_zero = True
                        continue
                    event_skip = True
                    skip_counts['too_short'] += 1
                    break

                # ── Resample to 50 Hz ──────────────────────────────────────────
                r = resample_stream(xyz, src_freq=native_freq)

                # ── Pad/trim to WINDOW_SIZE ────────────────────────────────────
                # This is the key fix: resampling rounding can yield 249
                # instead of 250 — pad with edge values rather than reject
                r_fixed = pad_or_trim(r, WINDOW_SIZE)
                if r_fixed is None:
                    # Genuinely too short (>20% missing)
                    if stream_name in OPTIONAL_STREAMS:
                        resampled[stream_name] = np.zeros((WINDOW_SIZE, 3), dtype=np.float32)
                        event_zero = True
                        continue
                    event_skip = True
                    skip_counts['too_short'] += 1
                    break

                if len(r) < WINDOW_SIZE:
                    event_padded = True
                    skip_counts['padded'] += 1

                resampled[stream_name] = r_fixed

            if event_skip:
                skip_counts['total'] += 1
                continue

            if event_zero:
                skip_counts['zero_filled'] += 1

            # ── All streams present — find minimum length for windowing ────────
            # streams are already padded/trimmed to WINDOW_SIZE, but longer
            # recordings are still their full resampled length
            # We need the actual resampled arrays (not all are WINDOW_SIZE)
            # Re-check: pad_or_trim trims longer arrays to WINDOW_SIZE too,
            # so all arrays are exactly WINDOW_SIZE here.
            # For longer recordings we want to slide — reload those streams.
            # OPTIMISATION: re-resample only if we need more than 1 window.
            # Simple approach: if any stream has >WINDOW_SIZE samples after
            # resampling, do proper windowing. Otherwise single window.
            # Since pad_or_trim already trimmed to WINDOW_SIZE, we have
            # exactly one window per event for short recordings.
            # For longer recordings (>WINDOW_SIZE after resampling), we need
            # to go back to the raw resampled data. Re-parse is expensive.
            # Pragmatic fix: store resampled result before pad_or_trim for
            # the reference stream, then apply windowing if possible.
            #
            # Actually pad_or_trim TRIMS longer arrays to WINDOW_SIZE —
            # this means long recordings only yield 1 window instead of many.
            # For Humcare, recordings are ~5-6 seconds = ~300-310 samples at 50Hz
            # → 1-2 windows with stride 125. This is acceptable given we have
            # 87 subjects × 38 activities and the skip rate is the bigger problem.
            # The windowing is already in the original resampled data before trimming.
            # Let's do it properly: re-resample and window the reference stream.

            # Quick path: if all resampled to ≥ WINDOW_SIZE, do proper windowing
            # We need to re-parse phone_accel (reference) for proper windowing
            ref_name   = STREAM_NAMES[0]   # phone_accel
            ref_native = STREAM_FREQS[ref_name]

            if ref_name in files:
                parsed_ref = parse_humcare_file(files[ref_name])
                if parsed_ref is not None:
                    _, xyz_ref = parsed_ref
                    r_ref = resample_stream(xyz_ref, src_freq=ref_native)
                    wins_ref = window_array(r_ref)
                else:
                    wins_ref = None
            else:
                wins_ref = None

            if wins_ref is None or len(wins_ref) == 0:
                # Fall back to single window from padded data
                n_wins = 1
                for sname in STREAM_NAMES:
                    arr = resampled[sname]  # already WINDOW_SIZE
                    batch_streams[sname].append(arr.reshape(1, WINDOW_SIZE, 3))
            else:
                n_wins = len(wins_ref)
                batch_streams[ref_name].append(wins_ref)

                # Window all other streams
                for sname in STREAM_NAMES[1:]:
                    native_f = STREAM_FREQS[sname]
                    if sname not in files or sname in OPTIONAL_STREAMS and sname not in files:
                        # Zero-fill for all windows
                        z = np.zeros((n_wins, WINDOW_SIZE, 3), dtype=np.float32)
                        batch_streams[sname].append(z)
                        continue
                    p2 = parse_humcare_file(files[sname]) if sname in files else None
                    if p2 is None:
                        z = np.zeros((n_wins, WINDOW_SIZE, 3), dtype=np.float32)
                        batch_streams[sname].append(z)
                        continue
                    _, xyz2 = p2
                    r2      = resample_stream(xyz2, src_freq=native_f)
                    r2f     = pad_or_trim(r2, len(r_ref))   # match ref length
                    if r2f is None:
                        r2f = np.zeros_like(r_ref)
                    w2 = window_array(r2f)
                    if w2 is None or len(w2) != n_wins:
                        # Fallback: tile the padded single window
                        w2 = np.tile(resampled[sname].reshape(1,WINDOW_SIZE,3), (n_wins,1,1))
                    batch_streams[sname].append(w2)

            batch_labels.extend([label]    * n_wins)
            batch_subj_ids.extend([subj_id] * n_wins)
            subj_wins    += n_wins
            total_windows += n_wins

    batch_count += 1

    # ── Progress every 5 subjects ──────────────────────────────────────────────
    if (s_idx+1) % 5 == 0 or (s_idx+1) == len(subjects_to_process):
        elapsed = time.time() - t_start
        rate    = (s_idx+1) / elapsed
        eta_min = (len(subjects_to_process)-s_idx-1) / rate / 60 if rate > 0 else 0
        total_skip = skip_counts['total']
        print(
            f'  [{s_idx+1:>3}/{len(subjects_to_process)}] '
            f'subj={subj_id:>3} '
            f'wins_this={subj_wins:>5} '
            f'total={total_windows:>7,} '
            f'skip={total_skip:>5} '
            f'pad={skip_counts["padded"]:>4} '
            f'zero={skip_counts["zero_filled"]:>4} '
            f'ETA={eta_min:.0f}min'
        )

    # ── Checkpoint every CKPT_EVERY subjects ──────────────────────────────────
    if batch_count >= CKPT_EVERY:
        save_checkpoint(batch_idx, batch_streams, batch_labels,
                        batch_subj_ids, skip_counts)
        print(f'  💾 Checkpoint {batch_idx} saved '
              f'({total_windows:,} total windows so far)')
        batch_streams  = {s: [] for s in STREAM_NAMES}
        batch_labels   = []
        batch_subj_ids = []
        batch_count    = 0
        batch_idx     += 1

# ── Save final partial batch ───────────────────────────────────────────────────
if batch_labels:
    save_checkpoint(batch_idx, batch_streams, batch_labels,
                    batch_subj_ids, skip_counts)
    print(f'  💾 Final checkpoint saved')

elapsed_total = (time.time()-t_start)/60
print(f'\n── Processing complete ──')
print(f'   Total windows    : {total_windows:,}')
print(f'   Skipped (reject) : {skip_counts["total"]:,}')
print(f'   Padded events    : {skip_counts["padded"]:,}  (short recording, edge-padded)')
print(f'   Zero-filled      : {skip_counts["zero_filled"]:,}  (missing optional glass stream)')
print(f'   Time elapsed     : {elapsed_total:.1f} min')


Checking for existing checkpoints...
   No checkpoints — starting fresh

Subjects remaining : 87
Window=250  Stride=125
Pad tolerance      : up to 50 samples (1.0s)
Optional streams   : {'glass_magnetometer', 'glass_gyro', 'glass_accel'}

  [  5/87] subj=  5 wins_this=  301 total=  1,130 skip=   62 pad=4498 zero= 280 ETA=1172min
  [ 10/87] subj= 10 wins_this=   28 total=  1,545 skip=   97 pad=6193 zero= 345 ETA=741min
  💾 Checkpoint 0 saved (1,545 total windows so far)
  [ 15/87] subj= 15 wins_this=  257 total=  2,591 skip=  219 pad=10439 zero= 481 ETA=800min
  [ 20/87] subj= 20 wins_this=   90 total=  3,392 skip=  310 pad=13934 zero= 562 ETA=741min
  💾 Checkpoint 1 saved (3,392 total windows so far)
  [ 25/87] subj= 25 wins_this=  337 total=  4,082 skip=  354 pad=16787 zero= 588 ETA=667min
  [ 30/87] subj= 30 wins_this=  194 total=  4,583 skip=  417 pad=19002 zero= 618 ETA=579min
  💾 Checkpoint 2 saved (4,583 total windows so far)
  [ 35/87] subj= 35 wins_this=    0 total=  5,558 skip

## Cell 8 — Merge Checkpoints into Final Arrays

Loads all checkpoint files and concatenates into final arrays.

In [ ]:
print('Merging all checkpoints...')
merged = load_all_checkpoints()

if merged is None:
    raise RuntimeError('No checkpoint files found — did Cell 7 run successfully?')

final_streams     = merged['streams']
final_y           = np.array(merged['labels'],      dtype=np.int64)
final_subject_ids = np.array(merged['subject_ids'], dtype=object)

N = final_y.shape[0]
print(f'── Final arrays ──')
for sname, arr in final_streams.items():
    print(f'   {sname:<22} : {arr.shape}')
print(f'   y              : {final_y.shape}  | classes: {len(np.unique(final_y))}')
print(f'   subject_ids    : {final_subject_ids.shape} | subjects: {len(np.unique(final_subject_ids))}')

# Sanity checks
ns = [v.shape[0] for v in final_streams.values()]
assert len(set(ns)) == 1, f'Stream length mismatch'
assert ns[0] == N
assert all(v.shape[1] == WINDOW_SIZE for v in final_streams.values())
assert all(v.shape[2] == 3 for v in final_streams.values())
print(f'\n✅ All checks passed — N={N:,} windows')

Merging all checkpoints...
── Final arrays ──
   phone_accel            : (15766, 250, 3)
   phone_gyro             : (15766, 250, 3)
   phone_magnetometer     : (15766, 250, 3)
   watch_accel            : (15766, 250, 3)
   watch_gyro             : (15766, 250, 3)
   watch_magnetometer     : (15766, 250, 3)
   glass_accel            : (15766, 250, 3)
   glass_gyro             : (15766, 250, 3)
   glass_magnetometer     : (15766, 250, 3)
   y              : (15766,)  | classes: 28
   subject_ids    : (15766,) | subjects: 85

✅ All checks passed — N=15,766 windows


## Cell 9 — Train / Val / Test Split

In [ ]:
all_subj_ids = sorted(np.unique(final_subject_ids).tolist())
train_subjects, val_subjects, test_subjects = split_subjects(all_subj_ids)

split_info = {
    'train': [int(s) for s in train_subjects],
    'val'  : [int(s) for s in val_subjects],
    'test' : [int(s) for s in test_subjects],
    'note' : 'Humcare: user-level split. Use subject_activity_map for episode construction.'
}

for split_name, subjects in [('train',train_subjects),('val',val_subjects),('test',test_subjects)]:
    mask    = get_split_mask(final_subject_ids, subjects)
    n_wins  = mask.sum()
    n_acts  = len(np.unique(final_y[mask]))
    print(f'   {split_name:<6}: {len(subjects):>2} subjects | {n_wins:>7,} windows | {n_acts:>2} activities')

print('\n✅ Split complete')

   train : 63 subjects |  12,042 windows | 28 activities
   val   : 10 subjects |   1,986 windows | 28 activities
   test  : 12 subjects |   1,738 windows | 28 activities

✅ Split complete


## Cell 10 — Per-Stream Normalisation

In [ ]:
normalizer_dir = os.path.join(PROCESSED_PATHS['humcare'], 'normalizers')
os.makedirs(normalizer_dir, exist_ok=True)

train_mask = get_split_mask(final_subject_ids, train_subjects)

print('Fitting normalisers on train subjects only...')
normalized_streams = {}
for stream_name, arr in final_streams.items():
    norm     = StreamNormalizer()
    norm.fit(arr[train_mask])
    arr_norm = norm.transform(arr)
    normalized_streams[stream_name] = arr_norm
    norm.save(os.path.join(normalizer_dir, stream_name))
    freq = STREAM_FREQS[stream_name]
    print(f'   {stream_name:<22} ({freq:>3}Hz→50Hz) '
          f'mean={arr_norm[train_mask].mean():.4f} '
          f'std={arr_norm[train_mask].std():.4f}')

print('\n✅ Normalisation done')

Fitting normalisers on train subjects only...
   phone_accel            (400Hz→50Hz) mean=-0.0009 std=1.0013
   phone_gyro             (400Hz→50Hz) mean=0.0000 std=1.0014
   phone_magnetometer     (100Hz→50Hz) mean=-0.0001 std=1.0000
   watch_accel            (100Hz→50Hz) mean=-0.0000 std=1.0069
   watch_gyro             (100Hz→50Hz) mean=-0.0000 std=1.0004
   watch_magnetometer     (100Hz→50Hz) mean=-0.0000 std=1.0059
   glass_accel            (  5Hz→50Hz) mean=0.0001 std=0.9986
   glass_gyro             (  5Hz→50Hz) mean=-0.0018 std=0.9992
   glass_magnetometer     (  5Hz→50Hz) mean=0.0003 std=1.0057

✅ Normalisation done


## Cell 11 — Save and Clean Up Checkpoints

In [ ]:
print('Saving processed Humcare AF dataset...')
save_processed_dataset(
    dataset_name         = 'humcare',
    stream_arrays        = normalized_streams,
    y                    = final_y,
    subject_ids          = final_subject_ids,
    stream_names         = STREAM_NAMES,
    subject_activity_map = subject_activity_map,
    split_info           = split_info,
)

# Delete checkpoint files now that the final dataset is saved
ckpt_files = [
    f for f in os.listdir(CKPT_DIR)
    if f.startswith('ckpt_batch_') and f.endswith('.npz')
]
for fname in ckpt_files:
    os.remove(os.path.join(CKPT_DIR, fname))
print(f'   Deleted {len(ckpt_files)} checkpoint file(s)')
print('✅ Saved and cleaned up')

Saving processed Humcare AF dataset...
   Saved stream phone_accel               → (15766, 250, 3)  →  /content/drive/MyDrive/fsl_har_processed/humcare/phone_accel.npy
   Saved stream phone_gyro                → (15766, 250, 3)  →  /content/drive/MyDrive/fsl_har_processed/humcare/phone_gyro.npy
   Saved stream phone_magnetometer        → (15766, 250, 3)  →  /content/drive/MyDrive/fsl_har_processed/humcare/phone_magnetometer.npy
   Saved stream watch_accel               → (15766, 250, 3)  →  /content/drive/MyDrive/fsl_har_processed/humcare/watch_accel.npy
   Saved stream watch_gyro                → (15766, 250, 3)  →  /content/drive/MyDrive/fsl_har_processed/humcare/watch_gyro.npy
   Saved stream watch_magnetometer        → (15766, 250, 3)  →  /content/drive/MyDrive/fsl_har_processed/humcare/watch_magnetometer.npy
   Saved stream glass_accel               → (15766, 250, 3)  →  /content/drive/MyDrive/fsl_har_processed/humcare/glass_accel.npy
   Saved stream glass_gyro                → (1

## Cell 12 — Verification

In [ ]:
print('Verifying saved dataset...\n')
loaded = load_processed_dataset('humcare')

print('── Shapes ──')
for name, arr in loaded['streams'].items():
    freq = STREAM_FREQS[name]
    print(f'   {name:<22} : {arr.shape}  (native {freq}Hz)')
print(f'   y              : {loaded["y"].shape}')

print('\n── Windows per activity (top 10) ──')
labels_found = sorted(np.unique(loaded['y']).tolist())
for label in labels_found[:10]:
    count = (loaded['y'] == label).sum()
    print(f'   [{label:>2}] {LABEL_TO_ACT.get(label,str(label)):<40}: {count:>5}')
if len(labels_found) > 10:
    print(f'   ... ({len(labels_found)} classes total)')

ns = [v.shape[0] for v in loaded['streams'].values()]
assert len(set(ns)) == 1
assert ns[0] == len(loaded['y'])
assert all(v.shape[1] == WINDOW_SIZE for v in loaded['streams'].values())
assert all(v.shape[2] == 3           for v in loaded['streams'].values())
assert loaded['subject_activity_map'] is not None
print('\n✅ All checks passed')

Verifying saved dataset...

✅ Loaded "humcare" — 15766 windows | 9 streams
── Shapes ──
   phone_accel            : (15766, 250, 3)  (native 400Hz)
   phone_gyro             : (15766, 250, 3)  (native 400Hz)
   phone_magnetometer     : (15766, 250, 3)  (native 100Hz)
   watch_accel            : (15766, 250, 3)  (native 100Hz)
   watch_gyro             : (15766, 250, 3)  (native 100Hz)
   watch_magnetometer     : (15766, 250, 3)  (native 100Hz)
   glass_accel            : (15766, 250, 3)  (native 5Hz)
   glass_gyro             : (15766, 250, 3)  (native 5Hz)
   glass_magnetometer     : (15766, 250, 3)  (native 5Hz)
   y              : (15766,)

── Windows per activity (top 10) ──
   [ 0] walking                                 :   989
   [ 1] slow_walk                               :  1063
   [ 3] jogging                                 :  1012
   [ 6] sitting                                 :  1028
   [ 7] standing                                :   636
   [ 8] laying                  

## Cell 13 — Final Summary

In [ ]:
N = loaded['y'].shape[0]
print('=' * 65)
print('  HUMCARE AF PREPROCESSING COMPLETE')
print('=' * 65)
print(f'  Total windows        : {N:,}')
print(f'  Streams              : {N_STREAMS}')
for s in STREAM_NAMES:
    freq = STREAM_FREQS[s]
    print(f'    {s:<24} (native {freq:>3}Hz → 50Hz)')
print(f'  Window shape         : ({WINDOW_SIZE}, 3) @ {TARGET_FREQ}Hz')
print(f'  Classes              : {len(labels_found)} activities')
print(f'  Subjects             : {len(all_subj_ids)}')
print(f'    Train : {len(train_subjects)} | Val: {len(val_subjects)} | Test: {len(test_subjects)}')
print(f'  Saved to             : {PROCESSED_PATHS["humcare"]}')
print('=' * 65)
print()
print('  ✅ All 4 datasets preprocessed')
print('  ✅ Ready for notebook 05 (CNN backbone training)')
print('=' * 65)

  HUMCARE AF PREPROCESSING COMPLETE
  Total windows        : 15,766
  Streams              : 9
    phone_accel              (native 400Hz → 50Hz)
    phone_gyro               (native 400Hz → 50Hz)
    phone_magnetometer       (native 100Hz → 50Hz)
    watch_accel              (native 100Hz → 50Hz)
    watch_gyro               (native 100Hz → 50Hz)
    watch_magnetometer       (native 100Hz → 50Hz)
    glass_accel              (native   5Hz → 50Hz)
    glass_gyro               (native   5Hz → 50Hz)
    glass_magnetometer       (native   5Hz → 50Hz)
  Window shape         : (250, 3) @ 50Hz
  Classes              : 28 activities
  Subjects             : 85
    Train : 63 | Val: 10 | Test: 12
  Saved to             : /content/drive/MyDrive/fsl_har_processed/humcare

  ✅ All 4 datasets preprocessed
  ✅ Ready for notebook 05 (CNN backbone training)
